In [ ]:
%matplotlib widget
import pandas 
import pathlib
import ipywidgets
from ipyfilechooser import FileChooser

import numpy

import read_h5

import magic_graphs
import scipp
import plopp
import widget_code_cell
from ui import spinner

import operations_with_da

In [ ]:
# Global
DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])

DG_DATA = scipp.DataGroup()

DG_CALC = scipp.DataGroup()


In [ ]:
# Functions
def file_metadata(path: pathlib.Path):
    return {
        "filename": path.name,
        "size_gb": round(path.stat().st_size / 1e9, 2),
        "modified": pandas.to_datetime(path.stat().st_mtime, unit="s"),
        "full_path": path.resolve(),
    }


def display_center(*graphs):
    return display(
        ipywidgets.VBox(graphs, layout=ipywidgets.Layout(align_items="center"))
    )

def assign_dg_to_da_coords(dg: scipp.DataGroup, da: scipp.DataArray, prefix:str=""):
    for s_key in dg.keys():
        da.coords[f"{prefix}_{s_key}"] = dg[s_key]

def move_data_from_dg_to_da(dg_magic, data_event):
    assign_dg_to_da_coords(dg_magic['sample'], data_event, prefix="sample")
    data_event.coords['tp_position'] = dg_magic['tp_position']
    data_event.coords['source_position'] = dg_magic['source_position']
    data_event.coords['delta_L'] = dg_magic['delta_L']
    data_event.coords['delta_t'] = dg_magic['delta_t']
    data_event = data_event.transform_coords(
    ("two_theta",), graph={**magic_graphs.graph_qvec, **magic_graphs.graph_detector}
    )
    return


def load_dg_by_df_row(df_row):
    file_path = df_row['full_path']
    d_out = read_h5.read_magic_from_nexus(file_path)
    s_key = str(df_row.name)
    DG_DATA[s_key] = d_out
    return
    

def take_dg_by_df_row(df_row):
    s_key = str(df_row.name)
    if not(s_key in DG_DATA.keys()):
        print(f"Loading file '{df_row['filename']}'.")
        load_dg_by_df_row(df_row)
    d_out = DG_DATA[s_key]
    return d_out


def take_name_from_dg_by_df_row(df_row, name:str):
    d_out = take_dg_by_df_row(df_row)
    res = d_out.get(name, None)
    return res


def take_dg_calc_by_df_row(df_row):
    s_key = str(df_row.name)
    if not(s_key in DG_CALC.keys()):
        DG_CALC[s_key] = scipp.DataGroup()
    return DG_CALC[s_key]


def take_detector_a_laue_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_laue_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_a = take_name_from_dg_by_df_row(df_row, 'detector_a')
        detector_a_laue_hist = operations_with_da.da_to_laue_hist(detector_a, factor_border=0.00)
        dg_calc[s_key] = detector_a_laue_hist
    return dg_calc[s_key]


def take_detector_b_laue_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_laue_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_b = take_name_from_dg_by_df_row(df_row, 'detector_b')
        detector_b_laue_hist = operations_with_da.da_to_laue_hist(detector_b, factor_border=0.00)
        dg_calc[s_key] = detector_b_laue_hist
    return dg_calc[s_key]


def take_detector_a_image_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_image'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_a = take_name_from_dg_by_df_row(df_row, 'detector_a')
        detector_a_image = operations_with_da.da_to_2d_hist(detector_a, factor_border=0.00)
        dg_calc[s_key] = detector_a_image
    return dg_calc[s_key]


def take_detector_b_image_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_image'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_b = take_name_from_dg_by_df_row(df_row, 'detector_b')
        detector_b_image = operations_with_da.da_to_2d_hist(detector_b, factor_border=0.00)
        dg_calc[s_key] = detector_b_image
    return dg_calc[s_key]


def take_detector_a_tof_tth_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_tof_tth_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")    
        dg_magic = take_dg_by_df_row(df_row)
        detector_a = dg_magic.get('detector_a', None)
        if detector_a is None:
            return
        move_data_from_dg_to_da(dg_magic, detector_a)
        detector_a_tof_tth = detector_a.transform_coords(
        ("two_theta",), graph={**magic_graphs.graph_qvec, **magic_graphs.graph_detector}
        )
        detector_a_tof_tth_hist = detector_a_tof_tth.hist(two_theta=120, toa=1000)
        dg_calc[s_key] = detector_a_tof_tth_hist
    return dg_calc[s_key]


def take_detector_a_tth_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_tth_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")    
        dg_magic = take_dg_by_df_row(df_row)
        detector_a = dg_magic.get('detector_a', None)
        if detector_a is None:
            return
        detector_a_tth_hist = detector_a.hist(toa=1000)
        dg_calc[s_key] = detector_a_tth_hist
    return dg_calc[s_key]


def take_detector_b_tth_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_tth_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")    
        dg_magic = take_dg_by_df_row(df_row)
        detector_b = dg_magic.get('detector_b', None)
        if detector_b is None:
            return
        detector_b_tth_hist = detector_b.hist(toa=1000)
        dg_calc[s_key] = detector_b_tth_hist
    return dg_calc[s_key]


def take_detector_a_peaks_gamma_nu_toa(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_peaks_gamma_nu_toa'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        dg_magic = take_dg_by_df_row(df_row)
        if dg_magic is None:
            return
        detector_a = dg_magic.get('detector_a', None)
        if detector_a is None:
            return
        detector_a_image = take_detector_a_image_by_df_row(df_row)
        np_toa, np_gamma, np_nu, sig_toa, sig_gamma, sig_nu = (
            operations_with_da.find_peaks_hist(detector_a_image, threshold=0.1)
        )
        dg_calc[s_key] = pandas.DataFrame(
            data={
                'toa': np_toa,
                'gamma': np_gamma,
                'nu': np_nu,
                'toa_sigma': sig_toa,
                'gamma_sigma': sig_gamma,
                'nu_sigma': sig_nu,
                })
    return dg_calc[s_key]


def take_detector_b_peaks_gamma_nu_toa(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_peaks_gamma_nu_toa'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        dg_magic = take_dg_by_df_row(df_row)
        if dg_magic is None:
            return
        detector_b = dg_magic.get('detector_b', None)
        if detector_b is None:
            return
        detector_b_image = take_detector_b_image_by_df_row(df_row)
        np_toa, np_gamma, np_nu, sig_toa, sig_gamma, sig_nu = (
            operations_with_da.find_peaks_hist(detector_b_image, threshold=0.1)
        )
        dg_calc[s_key] = pandas.DataFrame(
            data={
                'toa': np_toa,
                'gamma': np_gamma,
                'nu': np_nu,
                'toa_sigma': sig_toa,
                'gamma_sigma': sig_gamma,
                'nu_sigma': sig_nu,
                })
    return dg_calc[s_key]


In [ ]:
# Widgets

fc_path_input = FileChooser(
    path = pathlib.Path('.').resolve(),
    # show_only_dirs=True,
    title='',
    layout=ipywidgets.Layout(width="95%"),
    )


clean_button = ipywidgets.Button(
    description="Clean",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)


add_output = ipywidgets.Output()


file_selector = ipywidgets.SelectMultiple(
    options=DF_FILES["filename"].tolist(),
    description="",
    layout=ipywidgets.Layout(width="80%", height="200px")
)


filter = ipywidgets.Text(
    value="",
    placeholder="Filter",
    description="",
    layout=ipywidgets.Layout(width="80%")
)


plot_laue_button = ipywidgets.Button(
    description="3D Laue",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)

plot_detector_a_image_button = ipywidgets.Button(
    description="γ-ν-TOF",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)

plot_detector_a_tof_tth_button = ipywidgets.Button(
    description="TOF-2θ",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)

plot_detector_a_tof_button = ipywidgets.Button(
    description="TOF",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)

extra_plots_checkbox = ipywidgets.Checkbox(
    value=False,
    description='Options',
    disabled=False,
    indent=False
)

detector_type_radio_button = ipywidgets.RadioButtons(
    options=['Detector A', 'Detector B'],
    layout={'width': '158px', 'display': 'none'},
    description='',
    disabled=False
)

normalization_radio_button = ipywidgets.RadioButtons(
    options=['No Normalization', 'Per Monitor', 'Per Vanadium'],
    layout={'width': '150px', 'display': 'none'},
    description='',
    disabled=False
)


plot_graph_button = ipywidgets.Button(
    description="Display Graph",
    button_style="info",
    layout={'width': '150px', 'display': 'none'},
)

find_peaks_button = ipywidgets.Button(
    description="Find Peaks",
    button_style="success",
    layout=ipywidgets.Layout(width="150px")
)

load_output = ipywidgets.Output()

In [ ]:
# Callback
def refresh_selector():
    query = (filter.value or "").strip().lower()
    if DF_FILES.empty:
        file_selector.options = []
        return

    visible_indices = [
        idx for idx, name in DF_FILES["filename"].astype(str).items()
        if not query or query in name.lower()
    ]

    file_selector.options = [
        (DF_FILES.loc[idx, "filename"], idx) for idx in visible_indices
    ]


def on_filter_change(change):
    refresh_selector()


def on_add_clicked(b):
    add_output.clear_output()
    with add_output:
        p = pathlib.Path(fc_path_input.selected_path).expanduser()

        if not p.exists():
            print("Path does not exist:", p)
            return

        global DF_FILES

        # Helper: check if filename already exists in df
        def already_in_df(path):
            return path.resolve() in DF_FILES["full_path"].values

        # Directory → add all files
        if p.is_dir():
            files = [f for f in p.glob("*") if f.is_file() and f.suffix in ['.h5', '.nxs']]
            new_rows = []

            for f in files:
                if already_in_df(f):
                    print(f"Skipping (already in table): {f.name}")
                else:
                    new_rows.append(file_metadata(f))

            if new_rows:
                DF_FILES = pandas.concat([DF_FILES, pandas.DataFrame(new_rows)], ignore_index=True)
                print(f"Added {len(new_rows)} new files from directory:", p)
            else:
                print("No new files to add from directory.")

        # Single file → add only if not already present
        elif p.is_file():
            if already_in_df(p):
                print("File already in table:", p.name)
            else:
                DF_FILES = pandas.concat([DF_FILES, pandas.DataFrame([file_metadata(p)])], ignore_index=True)
                print("Added file:", p.name)
        else:
            print("Path is neither file nor directory:", p)
            return

        refresh_selector()


def on_clean_clicked(b):
    global DF_FILES
    add_output.clear_output()   
    load_output.clear_output()

    DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])
    DG_DATA.clear()
    DG_CALC.clear()
    refresh_selector()


def on_load_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Files loading...")
        display(spinner)
        spinner.layout.display = "inline-block"

        for _, row in selected_rows.iterrows():
            file_path = row['full_path']
            if not pathlib.Path(file_path).is_file():
                continue
            try:
                load_dg_by_df_row(row)
                print(f" - {row['filename']} (row index: {row.name})")
                print(f"   full_path: {row['full_path']}")
            except Exception as e:
                print(f"Error reading file: {e}")
                continue
        spinner.layout.display = "none"


def on_plot_laue_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting Laue Pattern...")
        display(spinner)
        spinner.layout.display = "inline-block"
        fig = None
        for _, row in selected_rows.iterrows():
            d_plot = {}
            vmax = 0.
            detector_a_laue_hist = take_detector_a_laue_hist_by_df_row(row)
            if not(detector_a_laue_hist is None):
                d_plot['detector_a'] = detector_a_laue_hist
                vmax = max([vmax, numpy.quantile(detector_a_laue_hist.data.values, 0.9)])
            detector_b_laue_hist = take_detector_b_laue_hist_by_df_row(row)
            if not(detector_b_laue_hist is None):
                d_plot['detector_b'] = detector_b_laue_hist
                vmax = max([vmax, numpy.quantile(detector_b_laue_hist.data.values, 0.9)])

            fig = plopp.scatter3d(
                d_plot,
                pos="event_position_global",
                cbar=True,
                size=0.005,
                opacity=0.75,
                vmax=vmax,
            )
            break
        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)


def on_plot_detector_a_image_clicked(b):
    load_output.clear_output()
    fig = None
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting Detector A Image...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            detector_a_image = take_detector_a_image_by_df_row(row)
            if detector_a_image is None:
                continue

            rad_to_deg = detector_a_image.assign_coords(
                event_gamma=detector_a_image.coords["event_gamma"].to(unit="deg"),
                event_nu=detector_a_image.coords["event_nu"].to(unit="deg"),
            )
            fig = plopp.inspector(
                rad_to_deg,
                dim="toa",
                orientation="vertical",
                logc=False,
                mode="rectangle",
                autoscale=True,
            )
            break
        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)


def on_plot_detector_a_tof_tth_clicked(b):
    load_output.clear_output()
    fig = None
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting Detector A TOF-2θ  Image...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            detector_a_tof_tth_hist = take_detector_a_tof_tth_hist_by_df_row(row)
            if detector_a_tof_tth_hist is None:
                continue
            rad_to_deg_hist = detector_a_tof_tth_hist.assign_coords(
                two_theta=detector_a_tof_tth_hist.coords["two_theta"].to(unit="deg"),
            )
            fig = plopp.plot(
                rad_to_deg_hist,
                coords=['toa', 'two_theta'],
            )            
            break
        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)

def on_plot_detector_a_tth_clicked(b):
    load_output.clear_output()
    fig = None
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting Detector A TOF-2θ  Image...")
        display(spinner)
        spinner.layout.display = "inline-block"
        d_plot = {}
        for _, row in selected_rows.iterrows():
            detector_a_tth_hist = take_detector_a_tth_hist_by_df_row(row)
            if detector_a_tth_hist is None:
                continue
            d_plot[str(row.name)] = detector_a_tth_hist
        spinner.layout.display = "none"
        fig = plopp.plot(
            d_plot, 
            coords=['toa', ],
        )            
        display_center(fig)

def toggle_extra_plots_checkbox(change):
    if change['new']:          # checkbox checked
        detector_type_radio_button.layout.display = 'block'
        normalization_radio_button.layout.display = 'block'
        plot_graph_button.layout.display = 'block'
    else:                      # checkbox unchecked
        detector_type_radio_button.layout.display = 'none'
        normalization_radio_button.layout.display = 'none'
        plot_graph_button.layout.display = 'none'


def on_plot_graph_clicked(b):
    load_output.clear_output()
    with load_output:
        fig = scipp.show_graph({**magic_graphs.graph_detector, **magic_graphs.graph_qvec, **magic_graphs.graph_hkl}, simplified=True)
        display(fig)


def on_find_peaks_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Searching peaks...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            print(f" - File '{row.name:}'")
            df_peaks_gamma_nu_toa = take_detector_a_peaks_gamma_nu_toa(row)
            if not(df_peaks_gamma_nu_toa is None):
                dg_magic = take_dg_by_df_row(row)
                detector_a = dg_magic.get('detector_a', None)
                detector_a = detector_a.transform_coords(
                    ("event_gamma", "event_nu", "event_position_global"),graph=magic_graphs.graph_detector, rename_dims=False,)
                range_sigma = 5
                operations_with_da.assign_event_peak_to_da(
                    detector_a, 
                    df_peaks_gamma_nu_toa["toa"], 
                    df_peaks_gamma_nu_toa["gamma"], 
                    df_peaks_gamma_nu_toa["nu"], 
                    df_peaks_gamma_nu_toa["toa_sigma"], 
                    df_peaks_gamma_nu_toa["gamma_sigma"], 
                    df_peaks_gamma_nu_toa["nu_sigma"], 
                    range_sigma)
                dg_magic['detector_a'] = detector_a
                print(f"   Number of found peaks on detector A is '{df_peaks_gamma_nu_toa["toa"].size:}'")
            
            df_peaks_gamma_nu_toa = take_detector_b_peaks_gamma_nu_toa(row)
            if not(df_peaks_gamma_nu_toa is None):
                dg_magic = take_dg_by_df_row(row)
                detector_b = dg_magic.get('detector_b', None)
                detector_b = detector_b.transform_coords(
                    ("event_gamma", "event_nu", "event_position_global"),graph=magic_graphs.graph_detector, rename_dims=False,)
                range_sigma = 5
                operations_with_da.assign_event_peak_to_da(
                    detector_b, 
                    df_peaks_gamma_nu_toa["toa"], 
                    df_peaks_gamma_nu_toa["gamma"], 
                    df_peaks_gamma_nu_toa["nu"], 
                    df_peaks_gamma_nu_toa["toa_sigma"], 
                    df_peaks_gamma_nu_toa["gamma_sigma"], 
                    df_peaks_gamma_nu_toa["nu_sigma"], 
                    range_sigma)
                dg_magic['detector_b'] = detector_b
                print(f"   Number of found peaks on detector B is '{df_peaks_gamma_nu_toa["toa"].size:}'")

        spinner.layout.display = "none"




In [ ]:
# Signals

fc_path_input.register_callback(on_add_clicked)
clean_button.on_click(on_clean_clicked)
filter.observe(on_filter_change, names='value')
plot_laue_button.on_click(on_plot_laue_clicked)
plot_detector_a_image_button.on_click(on_plot_detector_a_image_clicked)
plot_detector_a_tof_tth_button.on_click(on_plot_detector_a_tof_tth_clicked)
plot_detector_a_tof_button.on_click(on_plot_detector_a_tth_clicked)
extra_plots_checkbox.observe(toggle_extra_plots_checkbox, names='value')
plot_graph_button.on_click(on_plot_graph_clicked)

find_peaks_button.on_click(on_find_peaks_clicked)


In [ ]:
# Display

ipywidgets.VBox([
    fc_path_input, 
    add_output,
    ipywidgets.HBox([file_selector, clean_button, ]),
    filter,
    ipywidgets.HBox([ipywidgets.HTML("<b>Graphs: </b>", layout=ipywidgets.Layout(width="150px")), plot_laue_button, plot_detector_a_image_button, plot_detector_a_tof_tth_button, plot_detector_a_tof_button, extra_plots_checkbox]),
    ipywidgets.HBox([detector_type_radio_button, normalization_radio_button, plot_graph_button]),
    ipywidgets.HBox([ipywidgets.HTML("<b>Operations: </b>", layout=ipywidgets.Layout(width="150px")), find_peaks_button]),
    load_output
])

In [ ]:
widget_code_cell.make_code_cell(width="80%")